In [0]:
%pip install torch

# IMPORTS

In [0]:
import torch
print(torch.__version__)
print(f"GPU disponible: {torch.cuda.is_available()}")

In [0]:
import requests
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import collect_list,struct
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pyspark.ml.feature import MinMaxScaler, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import NumericType
from pyspark.sql.functions import col, isnan, when,lit
from pyspark.sql.types import NumericType
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data load

In [0]:
prices = spark.read.format('delta').load('/Volumes/market-mood/features/feature_store')

In [0]:
prices.printSchema()

# Normalization

In [0]:
numeric_cols = [c.name for c in prices.schema if isinstance(c.dataType, NumericType)]
numeric_cols

In [0]:
global_features = [ 'log_return',
 'rsi',
 'macd',
 'bb_position',
 'volatility',
 'vol_ratio']
ticker_features = ['close','volume']

In [0]:
prices = prices.dropna().toPandas()

In [0]:
scaler.fit_transform(prices[['rsi']])

In [0]:
scaler = StandardScaler()
for col in global_features:
    prices[col] = scaler.fit_transform(prices[[col]])



In [0]:
prices[global_features]

In [0]:
tickers = prices['ticker'].unique()

In [0]:
for ticker in tickers:
    for col in ticker_features:
        scaler = StandardScaler()
        ticker_prices = prices[prices['ticker'] == ticker]
        ticker_indexes = ticker_prices.index
        ticker_prices = scaler.fit_transform(ticker_prices[[col]])
    


In [0]:
display(ticker_prices)

In [0]:
for col in ticker_features:
    prices[col] = scaler.fit(prices[col]).transform(prices[col])

# Format data for entry

In [0]:
context_size = 20
w = Window.partitionBy('ticker').orderBy('date').rowsBetween(-(context_size-1),0)
prices = prices.withColumn('context',collect_list(struct('close','volume','log_return','rsi','macd','bb_position','volatility','vol_ratio')).over(w))


In [0]:
fs_pdf = prices.dropna().toPandas()#convertir a pandas para poder usarlo para entrenar
fs_pdf = fs_pdf.sort_values(["ticker", "date"]).reset_index(drop=True)

In [0]:
fs_pdf.iloc[0]['context']

In [0]:
print(f"Shape: {fs_pdf.shape}")
print(f"Tickers: {fs_pdf['ticker'].nunique()}")
print(f"Rango: {fs_pdf['date'].min()} → {fs_pdf['date'].max()}")

In [0]:
fs_pdf.info()
fs_pdf['date'] = pd.to_datetime(fs_pdf['date'])

In [0]:
FEATURE_COLS = ["close", "volume", "log_return", "rsi", "macd", 
                "bb_position", "volatility", "vol_ratio"]
TARGET_COL = "target"
SEQUENCE_LENGTH = 20

# Split temporal
train = fs_pdf[fs_pdf["date"] < "2023-01-01"]
val   = fs_pdf[(fs_pdf["date"] >= "2023-01-01") & (fs_pdf["date"] < "2024-01-01")]
test  = fs_pdf[fs_pdf["date"] >= "2024-01-01"]

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

In [0]:
def change_format(context_list):
    return np.array([[v for v in d.values()] for d in context_list])

# Convertir context a arrays (20, 8)
train["context"] = train["context"].map(change_format)
val["context"]   = val["context"].map(change_format)
test["context"]  = test["context"].map(change_format)

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

In [0]:
np.isnan(train.iloc[0]['context']).any()

In [0]:
train.info()


In [0]:
print(type(train["context"].iloc[0]))
print(train["context"].iloc[0])

In [0]:
# Ver cuántas secuencias tienen NaN
nan_count = train["context"].map(lambda x: np(x).any()).sum()
print(f"Secuencias con NaN: {nan_count}")

# Ver en qué features aparecen los NaN
sample = np.vstack(train["context"].values)
print(f"NaNs por columna: {np.isnan(sample).sum(axis=0)}")

In [0]:
scaler = StandardScaler()
scaler.fit(np.vstack(train["context"].values))
train["context"] = train["context"].map(lambda x: scaler.transform(x))
val["context"]   = val["context"].map(lambda x: scaler.transform(x))
test["context"]  = test["context"].map(lambda x: scaler.transform(x))


In [0]:
print("Normalización completada")
print(f"Media aproximada tras normalizar: {np.vstack(train['context'].values).mean():.4f}")
print(f"Std aproximada tras normalizar: {np.vstack(train['context'].values).std():.4f}")